<a href="https://colab.research.google.com/github/howsam/Building-a-ChatGPT-like-Model-from-Scratch/blob/main/Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  <font color='#FFE15D'><b>💎 Evaluation </b></font><font color='#FF0B55'></font>

# 🔴 **Environment Setup**

## 🟠 Change the font size of the output cells

In [ ]:
print('Salam Howsam!')

Salam Howsam!


In [ ]:
from IPython.display import HTML
shell = get_ipython()

def adjust_font_size():
  display(HTML('''<style>
    body {
      font-size: 20px;
    }
  '''))

if adjust_font_size not in shell.events.callbacks['pre_execute']:
  shell.events.register('pre_execute', adjust_font_size)

In [ ]:
print('Salam Howsam!')

Salam Howsam!


## 🟠 `pip`

In [ ]:
!pip install openai

# 🔴 **Import**

In [ ]:
import os
import io
import re
import math
import time
import yaml
import json
import random
import pathlib
import requests
from pprint import pprint
from tqdm.auto import tqdm
from collections import Counter
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from datasets import load_dataset
from tokenizers import Tokenizer

from openai import OpenAI

# 🔴 **Functions**

## 🟠 Model

In [ ]:
class MultiHeadAttention(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.n_embd = config.n_embd
        self.n_head = config.n_head
        self.head_size = self.n_embd // self.n_head

        self.qkv_proj = nn.Linear(self.n_embd, 3*self.n_embd, bias=False)
        self.c_proj = nn.Linear(self.n_embd, self.n_embd, bias=False)
        self.c_proj.residual = True

    def forward(self, x):
        B, T, C = x.shape
        # QKV linear
        q, k, v = self.qkv_proj(x).view(B, T, 3*self.n_head, self.head_size).transpose(1, 2).chunk(3, dim=-3)
        # Scaled Dot Product Attention using pytorch
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        # Reshape and final projection
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.c_proj(y)
        return y

In [ ]:
class FeedForward(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.n_embd = config.n_embd
        self.f_expnd = config.f_expnd

        self.up_proj = nn.Linear(self.n_embd, int(self.f_expnd*self.n_embd), bias=False)
        self.down_proj = nn.Linear(int(self.f_expnd*self.n_embd), self.n_embd, bias=False)
        self.down_proj.residual = True

    def forward(self, x):
        return self.down_proj(F.gelu(self.up_proj(x)))

In [ ]:
class DecoderBlock(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.n_embd = config.n_embd
        # Multi Head Attention
        self.ln1 = nn.LayerNorm(config.n_embd)
        self.mha = MultiHeadAttention(config)
        # Feed Forward Neural Network
        self.ln2 = nn.LayerNorm(config.n_embd)
        self.mlp = FeedForward(config)

    def forward(self, x):
        x = x + self.mha(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

In [ ]:
class GPT(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.config = config
        self.wte = nn.Embedding(config.vocab_size, config.n_embd) # Token embedding
        self.wpe = nn.Embedding(config.max_seq_len, config.n_embd) # Position embedding
        self.decoders = nn.ModuleList([DecoderBlock(config) for _ in range(config.n_layer)]) # Decoders
        self.lnf = nn.LayerNorm(config.n_embd)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False) # Classifier
        self.lm_head.weight = self.wte.weight # Weight tying

        self.apply(self._init_weights)

    def _init_weights(self, module):
        std = 0.02
        if isinstance(module, nn.Linear):
            if hasattr(module, 'residual'):
                std *= (2*self.config.n_layer)**-0.5
            nn.init.normal_(module.weight, mean=0.0, std=std)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=std)

    def forward(self, idx):
        B, T = idx.shape
        # Token Embedding + Position Embedding
        x = self.wte(idx) + self.wpe(torch.arange(T, device=idx.device))
        # Decoders
        for decoder in self.decoders:
            x = decoder(x)
        # Classifier
        x = self.lnf(x)
        logits = self.lm_head(x)
        return logits

In [ ]:
@dataclass
class GPTConfig:
    vocab_size: int = 50257
    max_seq_len: int = 1024
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768
    f_expnd: int = 4

## 🟠 Generate

In [ ]:
def generate(model, tokenizer, prompt, n_rep=5, max_seq_len=128, T=0.9, top_k=10, device='cuda', seed=42):
    # Get the token ID for <|endoftext|>
    eot_token_id = tokenizer.encode("<|endoftext|>").ids[0]

    # Tokenize the prompt and convert it to a tensor on the specified device (e.g., GPU)
    inputs = torch.tensor(tokenizer.encode(prompt).ids, dtype=torch.int, device=device)  # Shape: [T]
    n = inputs.shape[0]

    # Repeat the input prompt n_rep times to generate multiple sequences in parallel
    inputs = inputs.unsqueeze(0).repeat(n_rep, 1)  # Shape: [B, T] where B = n_rep

    # Set the model to evaluation mode
    model.eval()

    # Initialize a random number generator for sampling
    sample_rng = torch.Generator(device=device)
    sample_rng.manual_seed(seed)

    # Track which sequences are still active
    is_finished = torch.zeros(n_rep, dtype=torch.bool, device=device)

    # Disable gradient calculation for faster inference
    with torch.no_grad():
        # Continue generating tokens until reaching the maximum sequence length
        while inputs.shape[-1] < (max_seq_len + n):
            # Forward pass: get logits from the model
            logits = model(inputs)  # Shape: [B, T, vocab_size]

            # Apply temperature scaling and softmax to get probabilities for the next token
            probs = torch.softmax(logits[:, -1, :] / T, dim=-1)  # Shape: [B, vocab_size]

            # Select the top_k tokens with the highest probabilities
            topk_probs, topk_indices = torch.topk(probs, k=top_k, dim=-1)  # Shape: [B, top_k]

            # Sample one token from the top_k candidates based on their probabilities
            sampled = torch.multinomial(topk_probs, 1, generator=sample_rng)  # Shape: [B, 1]

            # Map the sampled indices back to the original token IDs
            next_token = torch.gather(topk_indices, -1, sampled).squeeze(-1) # Shape: [B]

            # For finished sequences, force pad with eot_token_id again to avoid changing inputs
            next_token = torch.where(is_finished, torch.tensor(eot_token_id, device=device), next_token)

            # Update finished mask
            is_finished = is_finished | (next_token == eot_token_id)

            # Append the sampled tokens to the input sequence
            inputs = torch.cat((inputs, next_token.unsqueeze(-1)), dim=-1)  # Shape: [B, T+1]

    # Decode the generated sequences back into text
    generated_text = tokenizer.decode_batch(inputs.tolist())

    # Slice off the prompt part, keep only the generated continuation
    only_generated = [gen[len(prompt):] for gen in generated_text]

    return only_generated

# 🔴 **GPT-Eval**

## 🟠 Config

In [ ]:
@dataclass
class GeneralConfig:
    tokenizer_path: str
    model_path: str
    llm_eval_name: str
    api_key: str
    prompts_yaml_url: str
    n_samples: int = 10
    max_new_tokens: int = 128
    temperature: float = 1.0
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

## 🟠 LLM Evaluator

In [ ]:
def deepseekv3_grade(prompt: str, completion: str):
    """
    Ask DeepSeek-V3 (model 'deepseek-chat') to grade a TinyStories completion.
    Returns a list in the order [grammar, creativity, consistency, plot].
    """
    delim = "***"
    full_text = f"{prompt.strip()} {delim} {completion.strip()}"

    system_msg = (
        "You are an English teacher grading a student's story completion. "
        "Return ONLY a JSON object with keys grammar, creativity, consistency, plot "
        "— each an integer 0-10."
    )

    user_msg = (
        "Here is the story. The part after the three asterisks (***) is the student's "
        "completion. Evaluate and respond ONLY with the JSON described.\n\n"
        f"{full_text}"
    )

    response = client.chat.completions.create(
        model = "deepseek-chat",
        messages = [
            {"role": "system", "content": system_msg},
            {"role": "user",   "content": user_msg}
        ],
        temperature = 0  # deterministic grading
    )

    # Extract JSON part (remove ```json ... ```)
    json_str = re.search(r"\{.*?\}", response.choices[0].message.content, re.DOTALL).group(0)

    # Parse and reorder the values
    grades_dict = json.loads(json_str)
    return [grades_dict[k] for k in ["grammar", "creativity", "consistency", "plot"]]

## 🟠 Evaluation

In [ ]:
# 0. General config

gen_cfg = GeneralConfig(
    tokenizer_path = 'data/bpe-tokenizer_tinystories.json',
    model_path = 'weights/model_hdim-512_layer-8_lossv129.pt',
    llm_eval_name = 'deepseek-chat',
    api_key = open("others/deepseek_api_key.txt", "r").read().strip(),
    max_new_tokens = 256,
    prompts_yaml_url = (
        "https://huggingface.co/datasets/roneneldan/TinyStories/"
        "resolve/main/Evaluation%20prompts.yaml"
    )
)

In [ ]:
# 1. Load prompts

data = requests.get(gen_cfg.prompts_yaml_url, timeout=30).text
prompts = yaml.safe_load(io.StringIO(data))
type(prompts), len(prompts)

(list, 44)

In [ ]:
prompts[-1]

'Once upon a time, there was a little boy who was always naughty. His mom was always telling him to be good, but he kept disobeying her rules and ignoring her warnings. \n\nOne day, he was so naughty that his mom decided to punish him. She told him that he had to'

In [ ]:
# 2. Load tokenizer

tokenizer = Tokenizer.from_file(gen_cfg.tokenizer_path)

In [ ]:
# 3. Load Pretrained GPT model

gpt_cfg = GPTConfig(
    vocab_size=10_000,
    max_seq_len=1024,
    n_layer=8,
    n_head=16,
    n_embd=512,
    f_expnd=4
)

model = GPT(gpt_cfg)
model.load_state_dict(torch.load(gen_cfg.model_path))
model.to(gen_cfg.device).eval()

GPT(
  (wte): Embedding(10000, 512)
  (wpe): Embedding(1024, 512)
  (decoders): ModuleList(
    (0-7): 8 x DecoderBlock(
      (ln1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (mha): MultiHeadAttention(
        (qkv_proj): Linear(in_features=512, out_features=1536, bias=False)
        (c_proj): Linear(in_features=512, out_features=512, bias=False)
      )
      (ln2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (mlp): FeedForward(
        (up_proj): Linear(in_features=512, out_features=2048, bias=False)
        (down_proj): Linear(in_features=2048, out_features=512, bias=False)
      )
    )
  )
  (lnf): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  (lm_head): Linear(in_features=512, out_features=10000, bias=False)
)

In [ ]:
# 4. Call DeepSeek API

client = OpenAI(api_key=gen_cfg.api_key, base_url="https://api.deepseek.com")

In [ ]:
# 5. Evaluate the quality of student GPT completions using the LLM teacher as a reference

results = []

for prompt_id, prompt in enumerate(tqdm(prompts, desc="Prompts")):
    # Generate
    completions = generate(
        model, tokenizer, prompt,
        n_rep=gen_cfg.n_samples, max_seq_len=256,
        T=gen_cfg.temperature, top_k=10
    )

    prompt_scores = []
    for i, completion in enumerate(completions):
        # Grade
        grades = deepseekv3_grade(prompt, completion)  # [grammar, ...]
        prompt_scores.append(grades)
        time.sleep(1.2)  # polite rate-limit; tweak as needed
    prompt_scores = np.array(prompt_scores)
    mean_scores   = prompt_scores.mean(axis=0)

    results.append({
        "prompt_id": prompt_id,
        "prompt": prompt,
        "completions": completions,
        "grammar": mean_scores[0],
        "creativity":mean_scores[1],
        "consistency":mean_scores[2],
        "plot": mean_scores[3]
    })

Prompts:   0%|          | 0/44 [00:00<?, ?it/s]

In [ ]:
df = pd.DataFrame(results)
overall = df.mean(numeric_only=True)

display(df.head())
display(overall)

,prompt_id,prompt,completions,grammar,creativity,consistency,plot
0,0,"Once upon a time, there lived a bunny in a fie...",[ barely move.\n\nLucy went to the doctor and ...,8.0,6.3,6.6,6.3
1,1,One day a girl walked into the living room and...,"[ she heard a voice coming from inside.\n\n""Pl...",7.9,6.6,6.6,6.5
2,2,"Once upon a time, there lived a hamster in the...",[ the mouse was stuck in the log.\n\nThe hamst...,6.8,5.9,7.3,6.6
3,3,Jack asked his mom if he could ride the bike a...,"[Be careful of the path, no matter how fast yo...",7.9,6.0,6.6,5.9
4,4,Alice was bored and wanted to find some advent...,[ go explore and see the world? It's so much f...,7.9,6.7,7.4,6.5


prompt_id      21.500000
grammar         7.811364
creativity      6.109091
consistency     6.859091
plot            6.288636
dtype: float64

In [ ]:
prompt_scores

[[9, 6, 8, 7]]

In [ ]:
pprint(prompt)
print()
pprint(completions[1])
print()
print(tokenizer.encode(completions[1]))

('Once upon a time, there lived a bunny in a field. Her name was Lucy. Lucy '
 'loved to have feasts and parties with her bunny friends. One day, when Lucy '
 "was about to leave for a feast at a friend's house, she realized she's "
 'starting to feel sick. She was so weak she could')

(' hardly run or jump!\n'
 '\n'
 'The other animals in the field noticed Lucy had gotten sick and decided to '
 'go see if she needed help. As they approached, they saw that she was lying '
 'on the ground and not moving. They knew she was too weak to do anything so '
 'they ran away and tried to make her move to the hospital. \n'
 '\n'
 "After a while, Lucy's bunny friends came back and brought her all the good "
 'news. They knew it must be important to get enough rest when it was a day to '
 "be healed. The moral of the story is: it's important to take care of "
 'yourself so you can heal the best of ones!')

Encoding(num_tokens=132, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_

In [ ]:
torch.cuda.empty_cache()